In [ ]:
# ---------------- code impératif avec sqlite3 -------------
"""
SQLITE3: BDD relationnelle avec un dialecte SQL simple
BDD en un seul fichier ou chargé en RAM
pas de host / port / user / password / droits
le module sqlite3 python est dans la lib standard

sqlite3 utilise la DbAPI python pour tous les clients de BDD en python
- méthode connect() => connexion (création ou connexion sur la bdd)
- méthode cursor()  => prompt sur la connexion
- méthode execute() => exécuter une requête SQL sur le cursor
- méthode fetch()   => faire sortir les enregistrements attendus
- méthode close()   => fermer la cnx
---------------- gérer les transactions -----------------
- méthode begin()   => commence une transaction
- methode commit()  => confitmer une tr
- méthode rollback()=> annuler une tr
"""

import sqlite3

conn = sqlite3.connect("dns.db")
cur  = conn.cursor()
req = "SELECT SQLITE_VERSION()"
cur.execute(req)
row  = cur.fetchone()
print(row)
conn.close()

# tuple à 1 élément vs un élément scalaire mais entouré de parenthèse
# s = ("bonjour",)
# i = (1)
# print(type(s), type(i))

In [ ]:
# ----------------- une connexion s'ouvre et se ferme ------------------
# exécuter un script SQL pour créer les tables de la bdd
with sqlite3.connect("dns.db") as conn:
    # hydratation des enrigestrement en Rows ~ dict
    conn.row_factory = sqlite3.Row
    cur  = conn.cursor()
    with open("./domain_names_sqlite3.sql", mode="r", encoding="utf-8") as f:
        script = f.read()
    cur.executescript(script)

    # tester le nb d'enregistrement de la table pays
    req = "SELECT COUNT(1) as nb FROM pays"
    cur.execute(req)
    row  = cur.fetchone()
    print(dict(row))

    

In [ ]:
# ----------------------- insérer 100k lignes à partir d'un csv -----------------
"""
1/ choper les 2 1ères colonnes du fichier dns_100000.csv
2/ utiliser une requête préparée : INSERT INTO domain_name (iso2,name) VALUES (?,?)
avec la bonne méthode pour l'exécuter avec les données

3/ pour aller plus loin: batching
exécuter 10 requêtes de 10000 valeurs
data =                                    [["XX", "xxxxxx"],[....],.....,[....]]
INSERT INTO domain_name (iso2,name) VALUES ('    XX','xxxxxx   '),('  'YY','yyyyy' ....),.....,('...        ')
HINT: commencer avec 10 valeurs !!!!
"""

import csv

data = []
with open("./data/dns_100000.csv", mode="r", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter=";")
    next(reader)
    for i, row in enumerate(reader, start=1):
        # if i > 10: break
        data.append(row[:2])

# for i in range(len(data)):
#     data[i] = "','".join(data[i])
# data = "'),('".join(data)

# data = "'),('".join(list(map(lambda r: "','".join(r), data)))
# req = f"INSERT INTO domain_name (iso2,name) VALUES ('{data}')"

try:
    with sqlite3.connect("dns.db") as conn:
        cur  = conn.cursor()
        
        ## version préparée
        # req = "INSERT INTO domain_name (iso2,name) VALUES (?,?)"
        # cur.executemany(req, data)

        # version batching
        while data:
            sample = data[:10**4]
            sample = "'),('".join(list(map(lambda r: "','".join(r), sample)))
            req = f"INSERT INTO domain_name (iso2,name) VALUES ('{sample}')"
            cur.execute(req)
            data = data[10**4:]
            # nb d'enregistrement changés
            print(cur.rowcount)
except (sqlite3.OperationalError, sqlite3.IntegrityError) as e:
    print(e)



In [ ]:
# ----------------- client sqlite3 en poo -------------------
"""
client sqlite en poo, consiste à une couche d'indirection, une couche d'isolation qui masque
l'utilisation du sql et autre détails liés à la techno
"""
import csv
from pathlib import Path
# annotations étendues
from typing import List

class SqliteClient:
    def __init__(self, db_path: str | Path =":memory:"):
        self.db_path = db_path

    def execute_script(self, script_path: str | Path) -> None:
        with sqlite3.connect("dns.db") as conn:
            cur  = conn.cursor()
            with open(script_path, mode="r", encoding="utf-8") as f:
                script = f.read()
            cur.executescript(script)

    def count(self, table: str) -> int:
        with sqlite3.connect("dns.db") as conn:
            cur  = conn.cursor()
            req = f"SELECT COUNT(1) FROM {table}"
            cur.execute(req)
            row  = cur.fetchone()
            return row[0]

    def insert(self, table: str, fields: List[str] | None, values: List[list]) -> int:
        with sqlite3.connect("dns.db") as conn:
            cur  = conn.cursor()
            fields_str = f"({",".join(fields)})" if fields is not None else ""
            req = f"INSERT INTO {table} {fields_str} VALUES (?,?)"
            cur.executemany(req, values)
            return cur.rowcount

if __name__ == '__main__':

    data = []
    with open("./data/dns_100000.csv", mode="r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter=";")
        next(reader)
        for i, row in enumerate(reader, start=1):
            # if i > 10: break
            data.append(row[:2])


    db = SqliteClient()
    db.execute_script("./domain_names_sqlite3.sql")
    print(db.count("pays"))
    print(db.insert("pays", ["iso2", "name"], data))

In [ ]:
# --------------- idem avec un gestionnaire de contexte ----------------------
"""
client sqlite en poo, consiste à une couche d'indirection, une couche d'isolation qui masque
l'utilisation du sql et autre détails liés à la techno
"""
import csv
from pathlib import Path
# annotations étendues
from typing import List

class SqliteClient:
    def __init__(self, db_path: str | Path =":memory:"):
        self.db_path = db_path

    def __enter__(self):
        """
        exécuté avec le with après le __init__
        """
        self.conn = sqlite3.connect(self.db_path)
        # l'objet retourné est la variable à droite de as
        return self

    def __exit__(self, exc_type, exc, tb):
        """
        - exécuté en sortant du bloc with

        - peut intercepter toute exception lancée dans le bloc
        """
        if exc:
            self.conn.rollback()
        else:
            self.conn.commit()
        self.conn.close()


    def execute_script(self, script_path: str | Path) -> None:
        cur  = self.conn.cursor()
        with open(script_path, mode="r", encoding="utf-8") as f:
            script = f.read()
        cur.executescript(script)

    def count(self, table: str) -> int:
        cur  = self.conn.cursor()
        req = f"SELECT COUNT(1) FROM {table}"
        cur.execute(req)
        row  = cur.fetchone()
        return row[0]

    def insert(self, table: str, fields: List[str] | None, values: List[list]) -> int:
        cur  = self.conn.cursor()
        fields_str = f"({",".join(fields)})" if fields is not None else ""
        req = f"INSERT INTO {table} {fields_str} VALUES (?,?)"
        cur.executemany(req, values)
        return cur.rowcount

    def close(self):
        self.conn.close()

if __name__ == '__main__':

    data = []
    with open("./data/dns_100000.csv", mode="r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter=";")
        next(reader)
        for i, row in enumerate(reader, start=1):
            # if i > 10: break
            data.append(row[:2])


    with SqliteClient("./dns.db") as db:
        db.execute_script("./domain_names_sqlite3.sql")
        print(db.count("pays"))
        print(db.insert("pays", ["iso2", "name"], data))

In [ ]:
# -------------------- initiation à sqlAlchemy (véritable ORM) ---------------

# pip install SQLAlchemy

# connexion et session pour pooling
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, Session

# définition des modèles
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
# champs SQL
from sqlalchemy import String, Enum, Integer, Float, DateTime, Text, ForeignKey, func

from pathlib import Path

# -------------------------------- GLOBALS ------------------------------------

DB_PATH = "./dns.db"
# chaîne de cnx :URI
DB_URI = f"sqlite:///{DB_PATH}"


engine = create_engine(
  DB_URI,
  # par défaut: on peut réutiliser le même thread pour une nouvelle cnx => pooling possible
  connect_args={"check_same_thread": False},
  # voir les requêtes SQL pour debug
  # echo=True
  implicit_returning=False
)

# factory de sessions
SessionLocal = sessionmaker(autoflush=False, autocommit=False, bind=engine)

# --------------------------- MODELES --------------------------------------------

class Base(DeclarativeBase):
  """classe de base pour tous les modèles (tables)"""
  pass

class DomainName(Base):
  __tablename__ = "domain_name"

  # la vraie colonne SQL s'appelle "dns_id" (cf. domain_names_sqlite3.sql)
  id: Mapped[int] = mapped_column("dns_id", primary_key=True, autoincrement=True)
  iso2: Mapped[str] = mapped_column(String(2), nullable=False)
  name: Mapped[str] = mapped_column(String(100), unique=True, nullable=False, index=True)

class Pays(Base):
  __tablename__ = "pays"
  iso2: Mapped[str] = mapped_column(String(2), primary_key=True)
  name: Mapped[str] = mapped_column(String(100), unique=True, nullable=False)

# ------------------------------- init_db --------------------------------------------------


def init_db():
  import re
  import csv

  Base.metadata.create_all(bind=engine)

  with SessionLocal() as session:
    # Import pays depuis le script SQL
    sql_path = "./domain_names_sqlite3.sql"
    with open(sql_path, "r", encoding="utf-8") as f:
      sql_content = f.read()

    match = re.search(r"INSERT INTO `pays` VALUES (.+);", sql_content)
    if match:
      pays_values = re.findall(r"\('([^']+)','([^']+)'\)", match.group(1))
      for iso2, name in pays_values:
        # Pays(iso2=iso2, name=name)  => objet enregistrement de table
        # cet objet a une réalité en python et aussi dans la base
        # il faut synchroniser régulièrement les objet py et sql flush() commit()
        session.merge(Pays(iso2=iso2, name=name))

    # Import domain_name depuis dns_10.csv
    csv_path = "./data/dns_100000.csv"
    with open(csv_path, mode="r", encoding="utf-8") as f:
      reader = csv.DictReader(f, delimiter=";")
      for row in reader:
        session.add(DomainName(name=row["Nom de domaine"], iso2=row["Pays BE"]))

    session.commit()

init_db()

C:\Users\Admin stagiaire.DESKTOP-8967908\AppData\Local\Temp\ipykernel_11252\1139007979.py:23: SADeprecationWarning: The create_engine.implicit_returning parameter is deprecated and will be removed in a future release. 
  engine = create_engine(


OperationalError: (sqlite3.OperationalError) no such column: id
[SQL: INSERT INTO domain_name (iso2, name) VALUES (?, ?) RETURNING id]
[parameters: ('FR', 'aaaaaaaaaaaaaaaaaa.fr')]
(Background on this error at: https://sqlalche.me/e/21/e3q8)

In [ ]:
# ------------------------- usage ----------------------------------------
from sqlalchemy import select, func

with SessionLocal() as session:
  # SELECT * FROM domain_name
  dns_count = session.execute(select(func.count()).select_from(DomainName)).scalar()
  print(f"Nombre de domaines: {dns_count}")